# 06 · Bangla voice: speaking and listening, offline

* **Output:** Meta's MMS-TTS Bengali (`facebook/mms-tts-ben`, CC-BY-NC 4.0:
  non-commercial use). It writes every Bangla prompt into
  `firmware/stm32h743/prompts/bn/`, so the **wearable and the laptop both speak
  Bangla**. A human recording with the same file name always wins: record
  `OBST_LEFT.wav` yourself and it is kept.
* **Input:** BanglaASR (Whisper fine-tuned on Bangla; faster-whisper for English) turns speech into
  text; `intent.route()` (the app's IntentRouter, in Python) turns text into
  an action. Mixed speech works: *"chair টা কোথায়"*.

Both models download once to the Hugging Face cache on C: and then need no network.

In [ ]:
import sys, pathlib
LAB = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(LAB))
from va_lab import config
config.ensure_dirs()
print("lab folder:", LAB)

In [ ]:
from va_lab import bangla_voice as bv, intent, audio_io
import sounddevice as sd
x = bv.synthesize("সামনে বাধা। দেড় মিটার।")
sd.play(x, 16000); sd.wait()

## Make (or refresh) every Bangla prompt
`overwrite=False` keeps existing files, including human recordings.

In [ ]:
n = bv.make_bangla_prompts(overwrite=False)
print("made", n, "new Bangla prompts in", config.PROMPT_WAVS / "bn")

In [ ]:
# Listen to a few through the assistant's own speaker
from va_lab.speech import Speaker, Utterance, ANSWER
spk = Speaker(bangla=True)
spk.say(Utterance(ANSWER, ["OBJ_CHAIR", "ZONE_LEFT", "DIST_1_5"])); import time; time.sleep(3)
spk.say(Utterance(ANSWER, ["ALARM_ON"])); time.sleep(4)
print("still English-only:", spk.missing_bangla())

## The router on text (no microphone)

In [ ]:
for u in ["চেয়ার টা কোথায়", "door koi", "সামনে কী আছে", "থামো", "জরুরি", "ব্যাটারি কত", "আজকে আবহাওয়া"]:
    print(f"{u:20s} -> {intent.route(u)}")

## Speak: run the cell, wait for the tone, say a command in Bangla (3 s)

In [ ]:
spk.say(Utterance(ANSWER, [("tone", "listen")])); time.sleep(0.3)
audio = audio_io.record(3.0)
text = bv.transcribe(audio, language="bn")
print("heard:", text)
print("action:", intent.route(text))

## How well it hears
Bangla uses **BanglaASR** (Whisper-small fine-tuned on Bangla). Tested on the
MMS voice: full sentences ("সামনে কী আছে", "দরজা খুঁজে দাও") come through;
single words ("থামো", "জরুরি") often do not. Those are what the keyword model
(notebook 03, key **K**) is for, and emergency is never matched loosely:
it needs the exact word or the alarm key.